# Milestone 5 - Fraud Detection & Behaviour Analysis

This notebook covers data exploration, behaviour feature engineering, Random Forest model training, and evaluation for behaviour-aware transaction fraud risk scoring.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# Load dataset
df = pd.read_csv('../data/transactions.csv')
print('Dataset Loaded Successfully!')

## 1. Data Exploration (EDA)
Inspect data structure, missing values, statistics, and fraud class distribution.

In [2]:
print('--- DF HEAD ---')
display(df.head())

print('\n--- DF SHAPE ---')
print(df.shape)

print('\n--- DF INFO ---')
df.info()

print('\n--- DF DESCRIBE ---')
display(df.describe())

print('\n--- MISSING VALUES ---')
print(df.isnull().sum())

print('\n--- FRAUD LABEL DISTRIBUTION ---')
print(df['fraud_label'].value_counts())

## 2. Behaviour Feature Engineering
Compute customer baselines (`user_avg_amount`), amount deviation ratios (`amount_deviation`), and time pattern anomalies (`time_deviation`).

In [3]:
# Datetime conversion
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M')
df = df.sort_values('timestamp').reset_index(drop=True)
df['hour'] = df['timestamp'].dt.hour

# Customer baseline average
df['user_avg_amount'] = df.groupby('customer_id')['amount'].transform(lambda x: x.expanding().mean().shift(1))
df['user_avg_amount'] = df['user_avg_amount'].fillna(df['amount'].median())

# Behaviour Deviation Ratios
df['amount_deviation'] = df['amount'] / (df['user_avg_amount'] + 1e-5)
df['is_suspicious_beneficiary'] = df['beneficiary_id'].astype(str).str.contains('SUSP', case=False).astype(int)
df['is_unusual_time'] = df['hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)

# Categorical Encoding
df = pd.get_dummies(df, columns=['merchant_category'], prefix='cat', drop_first=False)

display(df[['transaction_id', 'customer_id', 'amount', 'user_avg_amount', 'amount_deviation', 'fraud_label']].head(10))

## 3. Train Random Forest Model & Evaluate
Train Random Forest Classifier with balanced class weights to maximize Fraud Recall.

In [4]:
feature_cols = ['amount', 'amount_deviation', 'hour', 'is_unusual_time', 'is_suspicious_beneficiary'] + [c for c in df.columns if c.startswith('cat_')]
X = df[feature_cols]
y = df['fraud_label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print('--- CONFUSION MATRIX ---')
print(confusion_matrix(y_test, y_pred))

print('\n--- CLASSIFICATION REPORT ---')
print(classification_report(y_test, y_pred))

print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f} (Fraud Detection Target)')
print(f'F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')

## 4. Save Model Artifact
Serialize trained model to `../models/fraud_model.pkl` for FastAPI integration.

In [5]:
os.makedirs('../models', exist_ok=True)
artifact = {'model': rf_model, 'feature_cols': feature_cols}
joblib.dump(artifact, '../models/fraud_model.pkl')
print('Model saved to ../models/fraud_model.pkl')